<a href="https://colab.research.google.com/github/taanshubabariya-cyber/AIML-ML-LAB-03/blob/main/Mini_Project/dropout_risk_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
#      DROPOUT RISK DETECTION SYSTEM
#      Using Real Student Dataset (4424 records)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# STEP 1: Load Dataset
# ─────────────────────────────────────────────
print("=" * 65)
print("        DROPOUT RISK DETECTION SYSTEM")
print("        Real Student Dataset Analysis")
print("=" * 65)

df = pd.read_csv("/content/dataset.csv")

print(f"\n📂 Dataset Loaded Successfully!")
print(f"   Total Students  : {len(df)}")
print(f"   Total Features  : {df.shape[1] - 1}")
print(f"   Missing Values  : {df.isnull().sum().sum()}")

# ─────────────────────────────────────────────
# STEP 2: EDA
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 2: EXPLORATORY DATA ANALYSIS")
print("=" * 65)

print("\n📊 Target Class Distribution:")
counts = df["Target"].value_counts()
total  = len(df)
for label, cnt in counts.items():
    bar = "█" * int(cnt / 25)
    pct = cnt / total * 100
    print(f"   {label:<12} : {cnt:>4} students ({pct:5.1f}%)  {bar}")

print("\n📋 Sample Records (First 5):")
print(df[["Age at enrollment","Gender","Scholarship holder",
          "Debtor","Tuition fees up to date",
          "Curricular units 1st sem (approved)",
          "Curricular units 2nd sem (approved)","Target"]].head().to_string(index=False))

print("\n📈 Key Feature Statistics:")
key_cols = ["Age at enrollment",
            "Curricular units 1st sem (approved)",
            "Curricular units 1st sem (grade)",
            "Curricular units 2nd sem (approved)",
            "Curricular units 2nd sem (grade)",
            "Unemployment rate","GDP"]
print(df[key_cols].describe().round(2).to_string())

print("\n📊 Dropout Analysis by Key Factors:")

# Scholarship holder
print("\n  By Scholarship Holder:")
s = df.groupby(["Scholarship holder","Target"]).size().unstack(fill_value=0)
s["Dropout%"] = (s.get("Dropout",0) / s.sum(axis=1) * 100).round(1)
print(s.to_string())

# Debtor
print("\n  By Debtor Status:")
d = df.groupby(["Debtor","Target"]).size().unstack(fill_value=0)
d["Dropout%"] = (d.get("Dropout",0) / d.sum(axis=1) * 100).round(1)
print(d.to_string())

# Tuition fees
print("\n  By Tuition Fees Up To Date:")
t = df.groupby(["Tuition fees up to date","Target"]).size().unstack(fill_value=0)
t["Dropout%"] = (t.get("Dropout",0) / t.sum(axis=1) * 100).round(1)
print(t.to_string())

# Gender
print("\n  By Gender (0=Female, 1=Male):")
g = df.groupby(["Gender","Target"]).size().unstack(fill_value=0)
g["Dropout%"] = (g.get("Dropout",0) / g.sum(axis=1) * 100).round(1)
print(g.to_string())

# ─────────────────────────────────────────────
# STEP 3: Preprocessing — Binary (Dropout vs Non-Dropout)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 3: DATA PREPROCESSING")
print("=" * 65)

df_model = df.copy()

# Binary target: 1 = Dropout, 0 = Non-Dropout
df_model["Dropout"] = (df_model["Target"] == "Dropout").astype(int)
df_model.drop(columns=["Target"], inplace=True)

features = [c for c in df_model.columns if c != "Dropout"]
X = df_model[features]
y = df_model["Dropout"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n   Training samples : {len(X_train)}")
print(f"   Testing  samples : {len(X_test)}")
print(f"   Features used    : {len(features)}")
print(f"   Dropout in train : {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"   Dropout in test  : {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# ─────────────────────────────────────────────
# STEP 4: Train & Evaluate Models
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 4: MODEL TRAINING & EVALUATION")
print("=" * 65)

models = {
    "Logistic Regression":      ("scaled", LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest":            ("raw",    RandomForestClassifier(n_estimators=100, random_state=42)),
    "Gradient Boosting":        ("raw",    GradientBoostingClassifier(n_estimators=100, random_state=42)),
}

results = {}
for name, (dtype, model) in models.items():
    Xtr = X_train_sc if dtype == "scaled" else X_train
    Xte = X_test_sc  if dtype == "scaled" else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {"model": model, "pred": y_pred, "prob": y_prob,
                     "acc": acc, "auc": auc, "dtype": dtype}

    print(f"\n{'─'*45}")
    print(f"  🤖 {name}")
    print(f"{'─'*45}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  ROC-AUC   : {auc:.4f}")
    print(f"\n  Classification Report:")
    report = classification_report(y_test, y_pred,
                                   target_names=["Not Dropout","Dropout"])
    for line in report.splitlines():
        print("    " + line)
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n  Confusion Matrix:")
    print(f"    {'':15} Predicted No  Predicted Yes")
    print(f"    Actual No    :    TN={cm[0,0]:>4}         FP={cm[0,1]:>4}")
    print(f"    Actual Yes   :    FN={cm[1,0]:>4}         TP={cm[1,1]:>4}")

# ─────────────────────────────────────────────
# STEP 5: Feature Importance
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 5: TOP FEATURE IMPORTANCE (Random Forest)")
print("=" * 65)

rf = results["Random Forest"]["model"]
importance = pd.Series(rf.feature_importances_, index=features)
importance = importance.sort_values(ascending=False).head(15)

print("\n  Rank  Feature                                        Importance")
print("  " + "─" * 62)
for rank, (feat, imp) in enumerate(importance.items(), 1):
    bar = "█" * int(imp * 300)
    short = feat[:45]
    print(f"  {rank:>2}.  {short:<46} {imp:.4f}  {bar}")

# ─────────────────────────────────────────────
# STEP 6: Cross-Validation
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 6: 5-FOLD CROSS VALIDATION")
print("=" * 65)

print()
for name, info in results.items():
    Xdata = X_train_sc if info["dtype"] == "scaled" else X_train
    cv_scores = cross_val_score(info["model"], Xdata, y_train, cv=5, scoring="accuracy")
    print(f"  {name:<28} CV Acc: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

# ─────────────────────────────────────────────
# STEP 7: Predict on Sample Students from Dataset
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 7: PREDICT ON SAMPLE STUDENTS FROM DATASET")
print("=" * 65)

best_model_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_model_name]["model"]
sample = X_test.head(8).copy()
sample_preds = best_model.predict(sample)
sample_probs = best_model.predict_proba(sample)[:, 1]

def risk(p):
    if p >= 0.70: return "🔴 HIGH"
    if p >= 0.40: return "🟡 MEDIUM"
    return "🟢 LOW"

print(f"\n  Using Best Model: {best_model_name}")
print(f"\n  {'Student':<10} {'Dropout Prob':>14}  {'Risk Level':<16} {'Actual':<14} {'Match?'}")
print("  " + "─" * 65)
actual_vals = y_test.head(8).values
for i, (pred, prob, actual) in enumerate(zip(sample_preds, sample_probs, actual_vals), 1):
    actual_label = "Dropout" if actual == 1 else "Not Dropout"
    pred_label   = "Dropout" if pred  == 1 else "Not Dropout"
    match = "✅" if pred == actual else "❌"
    print(f"  Student {i:<3}  {prob*100:>11.1f}%  {risk(prob):<16} {actual_label:<14} {match}")

# ─────────────────────────────────────────────
# STEP 8: Risk Statistics
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 8: RISK LEVEL STATISTICS ON TEST SET")
print("=" * 65)

all_probs = results[best_model_name]["prob"]
high   = (all_probs >= 0.70).sum()
medium = ((all_probs >= 0.40) & (all_probs < 0.70)).sum()
low    = (all_probs < 0.40).sum()
total_test = len(all_probs)

print(f"\n  Total Test Students : {total_test}")
print(f"\n  🔴 HIGH   Risk (≥70%) : {high:>3} students ({high/total_test*100:.1f}%)  → Immediate Intervention")
print(f"  🟡 MEDIUM Risk (40-70%): {medium:>3} students ({medium/total_test*100:.1f}%)  → Monitor Closely")
print(f"  🟢 LOW    Risk (<40%) : {low:>3} students ({low/total_test*100:.1f}%)  → On Track")

# ─────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  FINAL SUMMARY")
print("=" * 65)

print(f"""
  📌 Project     : Dropout Risk Detection
  📌 Dataset     : 4,424 real students | 34 features
  📌 Target      : Binary (Dropout vs Non-Dropout)

  Model Performance Comparison:
  ┌────────────────────────────┬──────────┬──────────┐
  │ Model                      │ Accuracy │ ROC-AUC  │
  ├────────────────────────────┼──────────┼──────────┤
  │ Logistic Regression        │ {results['Logistic Regression']['acc']*100:>6.2f}%  │  {results['Logistic Regression']['auc']:.4f}  │
  │ Random Forest              │ {results['Random Forest']['acc']*100:>6.2f}%  │  {results['Random Forest']['auc']:.4f}  │
  │ Gradient Boosting          │ {results['Gradient Boosting']['acc']*100:>6.2f}%  │  {results['Gradient Boosting']['auc']:.4f}  │
  └────────────────────────────┴──────────┴──────────┘

  🏆 Best Model   : {best_model_name}
  🔑 Top Feature  : {importance.index[0]}
  ⚠️  Dropout Rate : {y.mean()*100:.1f}% of all students

  Key Dropout Risk Indicators:
  → Low curricular unit approvals (Sem 1 & 2)
  → Low grades in both semesters
  → Outstanding debts / tuition not up to date
  → No scholarship holder

  ✅ System successfully trained on your real dataset!
""")
print("=" * 65)

        DROPOUT RISK DETECTION SYSTEM
        Real Student Dataset Analysis

📂 Dataset Loaded Successfully!
   Total Students  : 4424
   Total Features  : 34
   Missing Values  : 0

  STEP 2: EXPLORATORY DATA ANALYSIS

📊 Target Class Distribution:
   Graduate     : 2209 students ( 49.9%)  ████████████████████████████████████████████████████████████████████████████████████████
   Dropout      : 1421 students ( 32.1%)  ████████████████████████████████████████████████████████
   Enrolled     :  794 students ( 17.9%)  ███████████████████████████████

📋 Sample Records (First 5):
 Age at enrollment  Gender  Scholarship holder  Debtor  Tuition fees up to date  Curricular units 1st sem (approved)  Curricular units 2nd sem (approved)   Target
                20       1                   0       0                        1                                    0                                    0  Dropout
                19       1                   0       0                        0              